# 04 - Prefill Conditions

Run all prefill conditions, loading models as needed.

**Phase 1 (PRIMARY_MODEL):** self_prefill, paraphrase_self, heavy_paraphrase_self, shuffled_steps, corrupted_numbers

**Phase 2 (CROSS_MODEL):** cross_prefill, paraphrase_cross, heavy_paraphrase_cross

The `normal` condition reuses answers from 02_generate_cots.
The `no_cot` condition was already run in 02_generate_cots.

**Fully resumable** - caches one file per (condition, problem).

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
from lib.data import load_gsm8k
from lib.prefill import run_prefill_condition

dataset = load_gsm8k()
print(f"Loaded {len(dataset)} GSM8K problems")

# Load all COTs
cot_lookup = {}
for p in COT_CACHE.glob("*.json"):
    r = json.loads(p.read_text())
    cot_lookup[r["problem_id"]] = r["cot_text"]
print(f"Loaded {len(cot_lookup)} COTs")

# Load all paraphrases/transforms
def load_transform_lookup(condition_prefix: str) -> dict:
    lookup = {}
    for p in PARAPHRASE_CACHE.glob(f"{condition_prefix}_*.json"):
        r = json.loads(p.read_text())
        lookup[r["problem_id"]] = r["paraphrased_cot"]
    return lookup

light_lookup = load_transform_lookup("paraphrase_light")
heavy_lookup = load_transform_lookup("paraphrase_heavy")
shuffle_lookup = load_transform_lookup("shuffle")
corrupt_lookup = load_transform_lookup("corrupt_numbers")

print(f"Light paraphrases: {len(light_lookup)}")
print(f"Heavy paraphrases: {len(heavy_lookup)}")
print(f"Shuffled: {len(shuffle_lookup)}")
print(f"Corrupted: {len(corrupt_lookup)}")

## Store `normal` condition results

These are just the COT generation answers from notebook 02 — no new inference.

In [ ]:
# Write normal condition results from COT cache (no inference needed)
for p in COT_CACHE.glob("*.json"):
    r = json.loads(p.read_text())
    pid = r["problem_id"]
    normal_path = PREFILL_CACHE / f"normal_{pid}.json"
    if not normal_path.exists():
        result = {
            "problem_id": pid,
            "condition": "normal",
            "model_used": PRIMARY_MODEL,
            "prefill_text": None,
            "predicted_answer": r["predicted_answer"],
            "gold_answer": r["gold_answer"],
            "generated_tokens": r["full_response"][-100:],
            "error": None,
        }
        normal_path.write_text(json.dumps(result))

print(f"Normal condition: {len(list(PREFILL_CACHE.glob('normal_*.json')))} cached")

## Phase 1: PRIMARY_MODEL conditions

self_prefill, paraphrase_self, heavy_paraphrase_self, shuffled_steps, corrupted_numbers

In [ ]:
from vllm import LLM, SamplingParams

print(f"Loading {PRIMARY_MODEL}...")
llm = LLM(model=PRIMARY_MODEL, dtype="bfloat16", max_model_len=4096)
print("PRIMARY_MODEL loaded.")

In [ ]:
# self_prefill: originating model reads its own verbatim COT
run_prefill_condition(llm, "self_prefill", PRIMARY_MODEL, dataset, cot_lookup)

In [ ]:
# paraphrase_self: originating model reads light-paraphrased COT
run_prefill_condition(llm, "paraphrase_self", PRIMARY_MODEL, dataset, light_lookup)

In [ ]:
# heavy_paraphrase_self: originating model reads heavy-paraphrased COT
run_prefill_condition(llm, "heavy_paraphrase_self", PRIMARY_MODEL, dataset, heavy_lookup)

In [ ]:
# shuffled_steps: originating model reads shuffled COT
run_prefill_condition(llm, "shuffled_steps", PRIMARY_MODEL, dataset, shuffle_lookup)

In [ ]:
# corrupted_numbers: originating model reads number-corrupted COT
run_prefill_condition(llm, "corrupted_numbers", PRIMARY_MODEL, dataset, corrupt_lookup)

In [ ]:
# Unload PRIMARY_MODEL
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print("PRIMARY_MODEL unloaded.")

## Phase 2: CROSS_MODEL conditions

cross_prefill, paraphrase_cross, heavy_paraphrase_cross

In [ ]:
print(f"Loading {CROSS_MODEL}...")
llm = LLM(model=CROSS_MODEL, dtype="bfloat16", max_model_len=4096)
print("CROSS_MODEL loaded.")

In [ ]:
# cross_prefill: different model reads verbatim COT
run_prefill_condition(llm, "cross_prefill", CROSS_MODEL, dataset, cot_lookup)

In [ ]:
# paraphrase_cross: different model reads light-paraphrased COT
run_prefill_condition(llm, "paraphrase_cross", CROSS_MODEL, dataset, light_lookup)

In [ ]:
# heavy_paraphrase_cross: different model reads heavy-paraphrased COT
run_prefill_condition(llm, "heavy_paraphrase_cross", CROSS_MODEL, dataset, heavy_lookup)

In [ ]:
# Unload CROSS_MODEL
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print("CROSS_MODEL unloaded.")

## Summary

In [ ]:
# Count cached results per condition
all_conditions = [
    "no_cot", "normal", "self_prefill", "cross_prefill",
    "paraphrase_self", "paraphrase_cross",
    "heavy_paraphrase_self", "heavy_paraphrase_cross",
    "shuffled_steps", "corrupted_numbers",
]

for cond in all_conditions:
    n = len(list(PREFILL_CACHE.glob(f"{cond}_*.json")))
    print(f"{cond:30s}: {n} results")

print("\nAll prefill conditions complete.")